# Fine-tuning Pre-trained Model for Perturbation Prediction

In [1]:
import json
import os
import sys
import time
import copy
from pathlib import Path
from typing import Iterable, List, Tuple, Dict, Union, Optional
import warnings
import scanpy as sc
import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

import torch
import numpy as np
import matplotlib
from torch import nn
from torch.nn import functional as F
from torch_geometric.loader import DataLoader
import pickle

# scGPT uses the GEARS backend
sys.path.insert(0, "/projects/p32655/irish/ExPert/Portable/GEARS/GEARS")
from gears import PertData, GEARS
from gears.inference import compute_metrics, deeper_analysis, non_dropout_analysis
from gears.utils import create_cell_graph_dataset_for_prediction

sys.path.insert(0, "/gpfs/projects/p32655/irish/CIPHER/Benchmarking/existing_models/scGPT")

import scgpt as scg
from scgpt.model import TransformerGenerator
from scgpt.loss import (
    masked_mse_loss,
    criterion_neg_log_bernoulli,
    masked_relative_error,
)
from scgpt.tokenizer import tokenize_batch, pad_batch, tokenize_and_pad_batch
from scgpt.tokenizer.gene_tokenizer import GeneVocab
from scgpt.utils import set_seed, map_raw_id_to_vocab_id, compute_perturbation_metrics

matplotlib.rcParams["savefig.transparent"] = False
warnings.filterwarnings("ignore")

set_seed(44)


/gpfs/projects/p32655/irish/.conda/envs/scGPT/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 ## Training Settings

In [2]:
# settings for data prcocessing
pad_token = "<pad>"
special_tokens = [pad_token, "<cls>", "<eoc>"]
pad_value = 0  # for padding values
pert_pad_id = 0
include_zero_gene = "all"
max_seq_len = 1536

# settings for training
MLM = True  # whether to use masked language modeling, currently it is always on.
CLS = False  # celltype classification objective
CCE = False  # Contrastive cell embedding objective
MVC = False  # Masked value prediction for cell embedding
ECS = False  # Elastic cell similarity objective
amp = True
load_model = "/projects/p32655/irish/CIPHER/Benchmarking/existing_models/scGPT/pretrained_models/scGPT_human"
load_param_prefixs = [
    "encoder",
    "value_encoder",
    "transformer_encoder",
]

# settings for optimizer
lr = 1e-4  # or 1e-4
batch_size = 64
eval_batch_size = 64
epochs = 15
schedule_interval = 1
early_stop = 10

# settings for the model
embsize = 512  # embedding dimension
d_hid = 512  # dimension of the feedforward network model in nn.TransformerEncoder
nlayers = 12  # number of nn.TransformerEncoderLayer in nn.TransformerEncoder
nhead = 8  # number of heads in nn.MultiheadAttention
n_layers_cls = 3
dropout = 0  # dropout probability
use_fast_transformer = True  # whether to use fast transformer

# logging
log_interval = 100

# dataset and evaluation choices
data_name = "akana_etal_2026_crispra_perturbseq"
# split = "simulation"
# if data_name == "norman":
#     perts_to_plot = ["SAMD1+ZBTB1"]
# elif data_name == "adamson":
#     perts_to_plot = ["KCTD16+ctrl"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [3]:
# save_dir = Path(f"./save/dev_perturb_{data_name}-{time.strftime('%b%d-%H-%M')}/")
# save_dir.mkdir(parents=True, exist_ok=True)

save_dir = "/gpfs/projects/p32655/irish/CIPHER/Benchmarking/results/scGPT/" + data_name
os.makedirs(save_dir, exist_ok=True)

print(f"saving to {save_dir}")

logger = scg.logger
scg.utils.add_file_handler(logger, save_dir + "/run.log")
# log running date and current git commit
logger.info(f"Running on {time.strftime('%Y-%m-%d %H:%M:%S')}")

saving to /gpfs/projects/p32655/irish/CIPHER/Benchmarking/results/scGPT/akana_etal_2026_crispra_perturbseq
scGPT - INFO - Running on 2026-07-07 22:57:01


In [4]:
# Load the full set
full_set = sc.read_h5ad("/projects/p32655/irish/CIPHER/Benchmarking/dataset_splits/" + data_name + "/filtered.h5ad")
logging.info("Loaded the full gene set")

data_folder = save_dir + "/saved_data"

control_indices = np.load("/projects/p32655/irish/CIPHER/Benchmarking/dataset_splits/" + data_name + "/control_idx.npy")
training_indices = np.load("/projects/p32655/irish/CIPHER/Benchmarking/dataset_splits/" + data_name + "/train_idx.npy")
testing_indices = np.load("/projects/p32655/irish/CIPHER/Benchmarking/dataset_splits/" + data_name + "/test_idx.npy")

control_set = full_set[control_indices]

full_set.obs.drop(columns=["condition"], errors="ignore", inplace=True)
full_set.obs.rename(columns={"perturbation": "condition"}, inplace=True)
# full_set.obs["condition"] = resolve_gene_symbols(full_set.obs["condition"])
full_set.obs.rename(columns={"celltype": "cell_type"}, inplace=True)

# GEARS Requires cell_type internally. As all datasets for the CIPHER benchmark are from a 
# single cell line, we can just set the dataset name as the cell type
if "cell_type" not in full_set.obs.columns:
    # scGPT's compute_perturbation_metrics parses condition_name (built as
    # "_".join([cell_type, condition, dose_val])) back into exactly 3 parts,
    # so cell_type itself must not contain underscores.
    full_set.obs["cell_type"] = data_name.replace("_", "-")
else:
    full_set.obs.rename(columns={"celltype": "cell_type"}, inplace=True)

perturbed_obs_names = full_set.obs_names
train_obs = perturbed_obs_names[training_indices]
id_test_obs = perturbed_obs_names[testing_indices]
    
# Restrict to the anndata to the cells in the control, train and test set. Other cells are unnecessary overhead
used_positions = np.unique(np.concatenate([control_indices, training_indices, testing_indices]))
full_set = full_set[used_positions].copy()
    
full_set.obs["condition"] = full_set.obs["condition"].apply(lambda x: "ctrl" if x == "control" else "ctrl+" + x)
print("Control cell types:", full_set.obs.loc[full_set.obs['condition'] == 'ctrl', 'cell_type'].unique())
print("Perturbed cell types:", full_set.obs.loc[full_set.obs['condition'] != 'ctrl', 'cell_type'].unique())
full_set.var["gene_name"] = full_set.var.index
logging.info("Reassigned annotated data columns to be compatible with GEARS")

# All 27 datasets are raw counts. Keep a raw-counts layer alongside the
# normalized X, matching run_CIPHER.py/run_scLAMBDA.py/run_scouter.py/
# run_GenePert.py, so the final scoring cell can report DELTA_X/CONTROL in
# raw space like the rest of the benchmark.
full_set.layers['counts'] = full_set.X.copy()
sc.pp.normalize_total(full_set)
sc.pp.log1p(full_set)
logging.info(f"Normalised the dataset")

pert_data = PertData(data_folder)
# scGPT never uses PertData's GO-graph perturbation embeddings (only GEARS' own
# GNN does), so the default curated "essential genes" list -- built from GEARS'
# CRISPRi essentiality-screen training data, not from this dataset -- has no
# benefit here and only drops perturbations whose target isn't on that list.
# default_pert_graph=False switches to (this panel's genes + all perturbation
# targets) intersected with GO annotations, which covers all of them.
pert_data.default_pert_graph = False
pert_data.new_data_process(dataset_name="pert_data", adata=full_set, save_anndata=False)
pert_data.load(data_path=data_folder + "/pert_data", adata=full_set)

train_conditions = list(set(full_set[train_obs].obs["condition"].tolist()))
id_test_conditions = list(set(full_set[id_test_obs].obs["condition"].tolist()))

split_dict = {
    'train': train_conditions,
    'val': train_conditions,  # Reusing the train set for the validation as there is no validation set. Risk of overfitting!
    'test': id_test_conditions,
}
print("Train perturbations:", len(split_dict['train']))
print("Val (ID) perturbations:", len(split_dict['val']))
print("Test (ID) perturbations:", len(split_dict['test']))

split_path = os.path.join(data_folder, "pert_data", "split.pkl")
with open(split_path, 'wb') as f:
    pickle.dump(split_dict, f)

pert_data.prepare_split(split='custom', split_dict_path=split_path)
pert_data.get_dataloader(batch_size=batch_size, test_batch_size=eval_batch_size)




# pert_data = PertData("./data")
# pert_data.load(data_name=data_name)
# pert_data.prepare_split(split=split, seed=1)
# pert_data.get_dataloader(batch_size=batch_size, test_batch_size=eval_batch_size)




2026-07-07 22:57:03,418 - INFO - Loaded the full gene set
2026-07-07 22:57:03,598 - INFO - Reassigned annotated data columns to be compatible with GEARS


Control cell types: ['akana-etal-2026-crispra-perturbseq']
Perturbed cell types: ['akana-etal-2026-crispra-perturbseq']


2026-07-07 22:57:03,851 - INFO - Normalised the dataset
Found local copy...


Found pyg file at /gpfs/projects/p32655/irish/CIPHER/Benchmarking/results/scGPT/akana_etal_2026_crispra_perturbseq/saved_data/pert_data/data_pyg/cell_graphs.pkl
The pyg file is 0.4 GB, so get comfy...


Done!
These perturbations are not in the GO graph and their perturbation can thus not be predicted
[]
Local copy of pyg dataset is detected. Loading...
Done!
Creating dataloaders....
Done!


Train perturbations: 57
Val (ID) perturbations: 57
Test (ID) perturbations: 57


In [5]:
if load_model is not None:
    model_dir = Path(load_model)
    model_config_file = model_dir / "args.json"
    model_file = model_dir / "best_model.pt"
    vocab_file = model_dir / "vocab.json"

    vocab = GeneVocab.from_file(vocab_file)
    for s in special_tokens:
        if s not in vocab:
            vocab.append_token(s)

    pert_data.adata.var["id_in_vocab"] = [
        1 if gene in vocab else -1 for gene in pert_data.adata.var["gene_name"]
    ]
    gene_ids_in_vocab = np.array(pert_data.adata.var["id_in_vocab"])
    logger.info(
        f"match {np.sum(gene_ids_in_vocab >= 0)}/{len(gene_ids_in_vocab)} genes "
        f"in vocabulary of size {len(vocab)}."
    )
    genes = pert_data.adata.var["gene_name"].tolist()

    # model
    with open(model_config_file, "r") as f:
        model_configs = json.load(f)
    logger.info(
        f"Resume model from {model_file}, the model args will override the "
        f"config {model_config_file}."
    )
    embsize = model_configs["embsize"]
    nhead = model_configs["nheads"]
    d_hid = model_configs["d_hid"]
    nlayers = model_configs["nlayers"]
    n_layers_cls = model_configs["n_layers_cls"]
else:
    genes = pert_data.adata.var["gene_name"].tolist()
    vocab = Vocab(
        VocabPybind(genes + special_tokens, None)
    )  # bidirectional lookup [gene <-> int]
vocab.set_default_index(vocab["<pad>"])
gene_ids = np.array(
    [vocab[gene] if gene in vocab else vocab["<pad>"] for gene in genes], dtype=int
)
n_genes = len(genes)



scGPT - INFO - match 4221/4273 genes in vocabulary of size 60697.
scGPT - INFO - Resume model from /projects/p32655/irish/CIPHER/Benchmarking/existing_models/scGPT/pretrained_models/scGPT_human/best_model.pt, the model args will override the config /projects/p32655/irish/CIPHER/Benchmarking/existing_models/scGPT/pretrained_models/scGPT_human/args.json.


 # Create and train scGpt

In [6]:
ntokens = len(vocab)  # size of vocabulary
model = TransformerGenerator(
    ntokens,
    embsize,
    nhead,
    d_hid,
    nlayers,
    nlayers_cls=n_layers_cls,
    n_cls=1,
    vocab=vocab,
    dropout=dropout,
    pad_token=pad_token,
    pad_value=pad_value,
    pert_pad_id=pert_pad_id,
    use_fast_transformer=use_fast_transformer,
)
if load_model is not None:
    pretrained_dict = torch.load(model_file)
    # scg.utils.load_pretrained rewrites the checkpoint's flash-attn-1.x-style
    # "self_attn.Wqkv.*" keys to this repo's FlashMHA wrapper layout
    # ("self_attn._impl.Wqkv.*") and drops any still-unmatched/shape-mismatched
    # keys instead of raising, unlike a raw model.load_state_dict().
    model = scg.utils.load_pretrained(
        model, pretrained_dict, strict=False, prefix=load_param_prefixs
    )
model.to(device)


scGPT - INFO - Loading parameter encoder.embedding.weight with shape torch.Size([60697, 512])
scGPT - INFO - Loading parameter encoder.enc_norm.weight with shape torch.Size([512])
scGPT - INFO - Loading parameter encoder.enc_norm.bias with shape torch.Size([512])
scGPT - INFO - Loading parameter value_encoder.linear1.weight with shape torch.Size([512, 1])
scGPT - INFO - Loading parameter value_encoder.linear1.bias with shape torch.Size([512])
scGPT - INFO - Loading parameter value_encoder.linear2.weight with shape torch.Size([512, 512])
scGPT - INFO - Loading parameter value_encoder.linear2.bias with shape torch.Size([512])
scGPT - INFO - Loading parameter value_encoder.norm.weight with shape torch.Size([512])
scGPT - INFO - Loading parameter value_encoder.norm.bias with shape torch.Size([512])
scGPT - INFO - Loading parameter transformer_encoder.layers.0.self_attn.in_proj_weight with shape torch.Size([1536, 512])
scGPT - INFO - Loading parameter transformer_encoder.layers.0.self_attn.

TransformerGenerator(
  (encoder): GeneEncoder(
    (embedding): Embedding(60697, 512, padding_idx=60694)
    (enc_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  )
  (value_encoder): ContinuousValueEncoder(
    (dropout): Dropout(p=0, inplace=False)
    (linear1): Linear(in_features=1, out_features=512, bias=True)
    (activation): ReLU()
    (linear2): Linear(in_features=512, out_features=512, bias=True)
    (norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  )
  (pert_encoder): Embedding(3, 512, padding_idx=0)
  (transformer_encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-11): 12 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=512, out_features=512, bias=True)
        )
        (linear1): Linear(in_features=512, out_features=512, bias=True)
        (dropout): Dropout(p=0, inplace=False)
        (linear2): Linear(in_features=512, out_features=512, bias=Tru

In [7]:

criterion = masked_mse_loss
criterion_cls = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, schedule_interval, gamma=0.9)
scaler = torch.cuda.amp.GradScaler(enabled=amp)


def train(model: nn.Module, train_loader: torch.utils.data.DataLoader) -> None:
    """
    Train the model for one epoch.
    """
    model.train()
    total_loss, total_mse = 0.0, 0.0
    start_time = time.time()

    num_batches = len(train_loader)
    for batch, batch_data in enumerate(train_loader):
        batch_size = len(batch_data.y)
        batch_data.to(device)
        x: torch.Tensor = batch_data.x  # (batch_size * n_genes, 2)
        ori_gene_values = x[:, 0].view(batch_size, n_genes)
        pert_flags = x[:, 1].long().view(batch_size, n_genes)
        target_gene_values = batch_data.y  # (batch_size, n_genes)

        if include_zero_gene in ["all", "batch-wise"]:
            if include_zero_gene == "all":
                input_gene_ids = torch.arange(n_genes, device=device, dtype=torch.long)
            else:
                input_gene_ids = (
                    ori_gene_values.nonzero()[:, 1].flatten().unique().sort()[0]
                )
            # sample input_gene_id
            if len(input_gene_ids) > max_seq_len:
                input_gene_ids = torch.randperm(len(input_gene_ids), device=device)[
                    :max_seq_len
                ]
            input_values = ori_gene_values[:, input_gene_ids]
            input_pert_flags = pert_flags[:, input_gene_ids]
            target_values = target_gene_values[:, input_gene_ids]

            mapped_input_gene_ids = map_raw_id_to_vocab_id(input_gene_ids, gene_ids)
            mapped_input_gene_ids = mapped_input_gene_ids.repeat(batch_size, 1)

            # src_key_padding_mask = mapped_input_gene_ids.eq(vocab[pad_token])
            src_key_padding_mask = torch.zeros_like(
                input_values, dtype=torch.bool, device=device
            )

        with torch.cuda.amp.autocast(enabled=amp):
            output_dict = model(
                mapped_input_gene_ids,
                input_values,
                input_pert_flags,
                src_key_padding_mask=src_key_padding_mask,
                CLS=CLS,
                CCE=CCE,
                MVC=MVC,
                ECS=ECS,
            )
            output_values = output_dict["mlm_output"]

            masked_positions = torch.ones_like(
                input_values, dtype=torch.bool
            )  # Use all
            loss = loss_mse = criterion(output_values, target_values, masked_positions)

        model.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        with warnings.catch_warnings(record=True) as w:
            warnings.filterwarnings("always")
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0,
                error_if_nonfinite=False if scaler.is_enabled() else True,
            )
            if len(w) > 0:
                logger.warning(
                    f"Found infinite gradient. This may be caused by the gradient "
                    f"scaler. The current scale is {scaler.get_scale()}. This warning "
                    "can be ignored if no longer occurs after autoscaling of the scaler."
                )
        scaler.step(optimizer)
        scaler.update()

        # torch.cuda.empty_cache()

        total_loss += loss.item()
        total_mse += loss_mse.item()
        if batch % log_interval == 0 and batch > 0:
            lr = scheduler.get_last_lr()[0]
            ms_per_batch = (time.time() - start_time) * 1000 / log_interval
            cur_loss = total_loss / log_interval
            cur_mse = total_mse / log_interval
            # ppl = math.exp(cur_loss)
            logger.info(
                f"| epoch {epoch:3d} | {batch:3d}/{num_batches:3d} batches | "
                f"lr {lr:05.4f} | ms/batch {ms_per_batch:5.2f} | "
                f"loss {cur_loss:5.2f} | mse {cur_mse:5.2f} |"
            )
            total_loss = 0
            total_mse = 0
            start_time = time.time()


def eval_perturb(
    loader: DataLoader, model: TransformerGenerator, device: torch.device
) -> Dict:
    """
    Run model in inference mode using a given data loader
    """

    model.eval()
    model.to(device)
    pert_cat = []
    pred = []
    truth = []
    pred_de = []
    truth_de = []
    results = {}
    logvar = []

    for itr, batch in enumerate(loader):
        batch.to(device)
        pert_cat.extend(batch.pert)

        with torch.no_grad():
            p = model.pred_perturb(
                batch,
                include_zero_gene=include_zero_gene,
                gene_ids=gene_ids,
            )
            t = batch.y
            pred.extend(p.cpu())
            truth.extend(t.cpu())

            # Differentially expressed genes
            for itr, de_idx in enumerate(batch.de_idx):
                pred_de.append(p[itr, de_idx])
                truth_de.append(t[itr, de_idx])

    # all genes
    results["pert_cat"] = np.array(pert_cat)
    pred = torch.stack(pred)
    truth = torch.stack(truth)
    results["pred"] = pred.detach().cpu().numpy().astype(np.float64)
    results["truth"] = truth.detach().cpu().numpy().astype(np.float64)

    pred_de = torch.stack(pred_de)
    truth_de = torch.stack(truth_de)
    results["pred_de"] = pred_de.detach().cpu().numpy().astype(np.float64)
    results["truth_de"] = truth_de.detach().cpu().numpy().astype(np.float64)

    return results



In [8]:
best_val_loss = float("inf")
best_val_corr = 0
best_model = None
patience = 0

for epoch in range(1, epochs + 1):
    epoch_start_time = time.time()
    train_loader = pert_data.dataloader["train_loader"]
    valid_loader = pert_data.dataloader["val_loader"]

    train(
        model,
        train_loader,
    )

    val_res = eval_perturb(valid_loader, model, device)
    val_metrics = compute_perturbation_metrics(
        val_res, pert_data.adata[pert_data.adata.obs["condition"] == "ctrl"]
    )
    logger.info(f"val_metrics at epoch {epoch}: ")
    logger.info(val_metrics)

    elapsed = time.time() - epoch_start_time
    logger.info(f"| end of epoch {epoch:3d} | time: {elapsed:5.2f}s | ")

    val_score = val_metrics["pearson"]
    if val_score > best_val_corr:
        best_val_corr = val_score
        best_model = copy.deepcopy(model)
        logger.info(f"Best model with score {val_score:5.4f}")
        patience = 0
    else:
        patience += 1
        if patience >= early_stop:
            logger.info(f"Early stop at epoch {epoch}")
            break

    # torch.save(
    #     model.state_dict(),
    #     save_dir / f"model_{epoch}.pt",
    # )

    scheduler.step()


scGPT - INFO - val_metrics at epoch 1: 
scGPT - INFO - {'pearson': 0.9816267101047452, 'pearson_de': 0.9045842378627423, 'pearson_delta': 0.038710434096994296, 'pearson_de_delta': 0.2418954959948776}
scGPT - INFO - | end of epoch   1 | time: 58.41s | 
scGPT - INFO - Best model with score 0.9816
scGPT - INFO - val_metrics at epoch 2: 
scGPT - INFO - {'pearson': 0.9918363686919898, 'pearson_de': 0.9272039619940825, 'pearson_delta': 0.08264802687659764, 'pearson_de_delta': 0.37156686929699445}
scGPT - INFO - | end of epoch   2 | time: 57.77s | 
scGPT - INFO - Best model with score 0.9918
scGPT - INFO - val_metrics at epoch 3: 
scGPT - INFO - {'pearson': 0.992407232929213, 'pearson_de': 0.9329457936349219, 'pearson_delta': 0.09131017806039796, 'pearson_de_delta': 0.5003350514382762}
scGPT - INFO - | end of epoch   3 | time: 57.67s | 
scGPT - INFO - Best model with score 0.9924
scGPT - INFO - val_metrics at epoch 4: 
scGPT - INFO - {'pearson': 0.9936211155823828, 'pearson_de': 0.93708890478

In [9]:
torch.save(best_model.state_dict(), save_dir + "/best_model.pt")


 ## Evaluations

In [12]:
def predict(
    model: TransformerGenerator, pert_list: List[str], pool_size: Optional[int] = None
) -> Dict:
    """
    Predict the gene expression values for the given perturbations.

    Args:
        model (:class:`torch.nn.Module`): The model to use for prediction.
        pert_list (:obj:`List[str]`): The list of perturbations to predict.
        pool_size (:obj:`int`, optional): For each perturbation, use this number
            of cells in the control and predict their perturbation results. Report
            the stats of these predictions. If `None`, use all control cells.
    """
    adata = pert_data.adata
    ctrl_adata = adata[adata.obs["condition"] == "ctrl"]
    if pool_size is None:
        pool_size = len(ctrl_adata.obs)
    gene_list = pert_data.gene_names.values.tolist()
    for pert in pert_list:
        for i in pert:
            if i not in gene_list:
                raise ValueError(
                    "The gene is not in the perturbation graph. Please select from GEARS.gene_list!"
                )

    model.eval()
    device = next(model.parameters()).device
    with torch.no_grad():
        results_pred = {}
        for pert in pert_list:
            cell_graphs = create_cell_graph_dataset_for_prediction(
                pert, ctrl_adata, gene_list, device, num_samples=pool_size
            )
            loader = DataLoader(cell_graphs, batch_size=eval_batch_size, shuffle=False)
            preds = []
            for batch_data in loader:
                pred_gene_values = model.pred_perturb(
                    batch_data, include_zero_gene, gene_ids=gene_ids, amp=amp
                )
                preds.append(pred_gene_values)
            preds = torch.cat(preds, dim=0)
            results_pred["_".join(pert)] = np.mean(preds.detach().cpu().numpy(), axis=0)

    return results_pred



In [13]:
from sklearn.metrics import mean_squared_error, r2_score

def to_dense(x):
    return np.asarray(x.toarray(), dtype=np.float32) if hasattr(x, "toarray") else np.asarray(x, dtype=np.float32)

ctrl_mask = pert_data.adata.obs["condition"] == "ctrl"
mean_expression = to_dense(pert_data.adata[ctrl_mask].X).mean(axis=0)

raw_ctrl_mean = to_dense(pert_data.adata[ctrl_mask].layers["counts"]).mean(axis=0)

# id_test_conditions are "ctrl+GENE"
id_test_pert_list = [cond[len("ctrl+"):].split("+") for cond in id_test_conditions]
predicted_means_dict = predict(best_model, id_test_pert_list, pool_size=300)

per_pert_corr = {}
per_pert_mse = {}
per_pert_r2 = {}
delta_y = {}
delta_x = {}
control = {}
skipped_perts = []

for cond, pert_genes in zip(id_test_conditions, id_test_pert_list):
    pred_centered = predicted_means_dict["_".join(pert_genes)] - mean_expression

    test_cells = pert_data.adata[pert_data.adata.obs["condition"] == cond]
    if test_cells.shape[0] == 0:
        skipped_perts.append(cond)
        continue
    gt_centered = to_dense(test_cells.X).mean(axis=0) - mean_expression

    with np.errstate(invalid="ignore", divide="ignore"):
        per_pert_corr[cond] = np.corrcoef(gt_centered, pred_centered)[0, 1]
    per_pert_mse[cond] = mean_squared_error(gt_centered, pred_centered)
    per_pert_r2[cond] = r2_score(gt_centered, pred_centered)

    # predicted gene expression change over the control in the log space
    delta_y[cond] = pred_centered

    # ground truth gene expression change over the control in raw space
    delta_x[cond] = to_dense(test_cells.layers["counts"]).mean(axis=0) - raw_ctrl_mean

    # mean control gene expression in raw space
    control[cond] = raw_ctrl_mean

if skipped_perts:
    print(f"Skipped {len(skipped_perts)} perturbation(s) with no test cells: {skipped_perts}")

nan_perts = sorted(p for p, v in per_pert_corr.items() if np.isnan(v))
if nan_perts:
    print(f"WARNING: {len(nan_perts)} perturbations have NaN correlation "
          f"(likely a zero-variance ground-truth or predicted profile): {nan_perts}")

mean_corr = np.nanmean(list(per_pert_corr.values()))
mean_mse = np.nanmean(list(per_pert_mse.values()))
mean_r2 = np.nanmean(list(per_pert_r2.values()))

exported_results = {"CORRELATION": per_pert_corr,
                     "MSE": per_pert_mse,
                     "R2": per_pert_r2,
                     "DELTA_Y": delta_y,
                     "CONTROL": control,
                     "DELTA_X": delta_x}

with open(save_dir + "/results.pkl", "wb") as file:
    pickle.dump(exported_results, file)

print(f"mean Pearson: {mean_corr:.4f} | mean MSE: {mean_mse:.4f} | mean R²: {mean_r2:.4f} over {len(per_pert_corr)} perturbations")


mean Pearson: 0.1487 | mean MSE: 0.0068 | mean R²: -0.3423 over 57 perturbations
